In [25]:
import torch
import torch.nn as nn
import numpy as np
from DDBSCAN import Raster_DBSCAN
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim
from Models import *
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.colors as mcolors
from ModelSocial import ImprovedSocialLSTM
import h5py

# from TrainSocialLSTM import extract_trajectories,prepare_training_data
# times new roman font
plt.rcParams["font.family"] = "Times New Roman"
seed = 414
# np.random.seed(seed)
colors = np.random.rand(600, 3)
colors = np.concatenate([np.array([[0,0,0]]),colors],axis = 0)
colormap = mcolors.ListedColormap(colors)

In [22]:
def prepare_numerical_data(traffic_context, batch_size=4):
    """
    Prepare numerical data from traffic context for the simplified model
    
    Args:
        traffic_context: Occupancy grid [batch, lane_cell_num, seq_length]
                         Each element is trajectory ID (0 = unoccupied)
        batch_size: Desired batch size
        
    Returns:
        inputs: Input tensor [batch_size, input_frames, 3]
        targets: Target positions [batch_size, output_size]
    """
    batch = traffic_context.size(0)
    lane_cells = traffic_context.size(1)
    seq_length = traffic_context.size(2)
    
    # Set parameters
    input_frames = 10
    window_size = input_frames + 1  # +1 for target
    
    inputs_list = []
    targets_list = []
    
    # Process each scenario in the batch
    for b in range(batch):
        # Find unique vehicle IDs
        unique_ids = torch.unique(traffic_context[b])
        unique_ids = unique_ids[unique_ids > 0]
        
        # Process each vehicle trajectory
        for vehicle_id in unique_ids:
            # Find positions of this vehicle at each time step
            positions = []
            for t in range(seq_length):
                # Find cells occupied by this vehicle
                cells = torch.nonzero(traffic_context[b, :, t] == vehicle_id, as_tuple=True)[0]
                if len(cells) > 0:
                    positions.append(cells[0].item())
                else:
                    positions.append(-1)
            
            # Create sliding windows
            for start_idx in range(seq_length - window_size + 1):
                window = positions[start_idx:start_idx + window_size]
                
                # Skip windows with missing positions
                if -1 in window:
                    continue
                
                # Extract input and target positions
                input_positions = window[:input_frames]
                target_position = window[input_frames]
                
                # Create input tensor with [position, front_dist, back_dist]
                input_data = torch.zeros(input_frames, 3)
        
                for t, pos in enumerate(input_positions):
                    # Set tracked vehicle position
                    input_data[t, 0] = pos

                    # Find front vehicle
                    front_dist = -999 
                    for front_pos in range(pos - 1, -1, -1):
                        curr_id = traffic_context[b, front_pos, start_idx + t]
                        if curr_id > 0 and curr_id != vehicle_id:
                            front_dist = front_pos - pos  # Will be negative
                            break
                    back_dist = 999
                    for back_pos in range(pos + 1, lane_cells):
                        curr_id = traffic_context[b, back_pos, start_idx + t]
                        if curr_id > 0 and curr_id != vehicle_id:
                            back_dist = back_pos - pos
                            break
                    input_data[t, 1] = front_dist 
                    input_data[t, 2] = back_dist

                
                inputs_list.append(input_data)
                targets_list.append(torch.tensor([target_position]))
                
                # If we have enough samples for a batch, yield them
                if len(inputs_list) >= batch_size:
                    inputs_batch = torch.stack(inputs_list[:batch_size])
                    targets_batch = torch.stack(targets_list[:batch_size])
                    
                    inputs_list = inputs_list[batch_size:]
                    targets_list = targets_list[batch_size:]
                    
                    yield inputs_batch, targets_batch
    
    # Yield any remaining samples
    if len(inputs_list) > 0:
        inputs_batch = torch.stack(inputs_list)
        targets_batch = torch.stack(targets_list)
        yield inputs_batch, targets_batch

In [28]:
out_folder = r"D:\TimeSpaceDiagramDataset\SocialLSTMDataset"
val_folder = os.path.join(out_folder,"val")
batch_size = 4
time_span = 100
val_dataset = TrajDataset(r"D:\TimeSpaceDiagramDataset\EncoderDecoder_EvenlySampled_FreeflowAug_0914_5res_lanechange_signal\100_frame\val",time_span)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
# find out the number of 1 over the total number of elements
counts = 0
for batch_idx,batch in enumerate(tqdm(val_loader)):
    traj_id = batch['traj_id']
    gen = prepare_numerical_data(traj_id)
    break

  0%|          | 0/61500 [00:02<?, ?it/s]


In [60]:
out_folder = r"D:\TimeSpaceDiagramDataset\SocialLSTMDataset"
val_folder = os.path.join(out_folder,"val")
os.makedirs(val_folder,exist_ok=True)
train_folder = os.path.join(out_folder,"train")
os.makedirs(train_folder,exist_ok=True)
train_count = 100000
val_count = 12000
batch_size = 4
p = 0.2
time_span = 100
val_dataset = TrajDataset(r"D:\TimeSpaceDiagramDataset\EncoderDecoder_EvenlySampled_FreeflowAug_0914_5res_lanechange_signal\100_frame\val",time_span)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
# find out the number of 1 over the total number of elements
counts = 0
h5_file = h5py.File(os.path.join(val_folder,"val_dataset.h5"),'w')
for batch_idx,batch in enumerate(tqdm(val_loader)):
    traj_id = batch['traj_id']
    gen = prepare_numerical_data(traj_id)
    # Create a group for this batch
    batch_group = h5_file.create_group(f'batch_{batch_idx}')
    batch_count = 0
    while True:
        try:
            inputs, targets = next(gen)
            if np.random.rand() > p:
                continue
            # inputs: [batch_size, input_frames, 3 (pos, front_dist, back_dist)], targets: [batch_size, 1]
            mini_batch = batch_group.create_group(f'mini_batch_{batch_count}')
            mini_batch.create_dataset('inputs', data=inputs.numpy(), compression="gzip")
            mini_batch.create_dataset('targets', data=targets.numpy(), compression="gzip")
            counts += inputs.size(0)
            batch_count += 1
        except StopIteration:
            break
    batch_group.attrs['total_samples'] = batch_count
    if counts > val_count:
        break
h5_file.attrs['total_trajectory_samples'] = counts
h5_file.close()

train_dataset = TrajDataset(r"D:\TimeSpaceDiagramDataset\EncoderDecoder_EvenlySampled_FreeflowAug_0914_5res_lanechange_signal\100_frame\train",time_span)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=1)
# find out the number of 1 over the total number of elements
counts = 0
h5_file = h5py.File(os.path.join(train_folder,"train_dataset.h5"),'w')
for batch_idx,batch in enumerate(tqdm(train_loader)):
    traj_id = batch['traj_id']
    gen = prepare_numerical_data(traj_id)
    # Create a group for this batch
    batch_group = h5_file.create_group(f'batch_{batch_idx}')
    batch_count = 0
    while True:
        try:
            inputs, targets = next(gen)
            if np.random.rand() > p:
                continue
            mini_batch = batch_group.create_group(f'mini_batch_{batch_count}')
            mini_batch.create_dataset('inputs', data=inputs.numpy(), compression="gzip")
            mini_batch.create_dataset('targets', data=targets.numpy(), compression="gzip")
            counts += inputs.size(0)
            batch_count += 1
        except StopIteration:
            break
    batch_group.attrs['total_samples'] = batch_count
    if counts > train_count:
        break
h5_file.attrs['total_trajectory_samples'] = counts
h5_file.close()



  0%|          | 456/246000 [55:30<498:06:42,  7.30s/it]


In [57]:
from Dataset import TrajectoryDataset

In [89]:
train_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\train\train_dataset.h5')
val_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\val\val_dataset.h5')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=1)
for batch in train_loader:
    inputs = batch['inputs']
    targets = batch['targets']
    break

In [32]:
from ModelSocial import ImprovedSocialLSTM
from TrainSocialLSTM import ConfidenceLoss
from Dataset import TrajectoryDataset
import os

# Training parameters
patience = 8 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 32
lane_unit = 200  # each lane unit is 0.5 meters
time_span = 100
hidden_size = 64
social_size = 32
num_layers = 2
learning_rate = 1e-4
weight_decay = 1e-5
dropout = 0.3
num_epochs = 100

criterion = ConfidenceLoss().to(device)
model = ImprovedSocialLSTM(
    hidden_size=hidden_size,
    social_size=social_size,
    num_layers=num_layers,
    input_frames=10,
    output_size=2,  # [position, confidence]
    dropout=dropout,
    device=device
)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
train_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\train\train_dataset.h5')
val_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\val\val_dataset.h5')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=1)
for batch in train_loader:
    inputs = batch['inputs'].to(device)
    targets = batch['targets'].to(device)
    break

In [233]:
outputs = model(inputs)

In [234]:
loss_dict = criterion(outputs, targets)

In [5]:
from ModelSocial import ImprovedSocialLSTM
from TrainSocialLSTM import ConfidenceLoss
import os
import json

In [28]:
model_path = r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\models\social_lstm\train_4\best_model.pth'
hyper_param_path = r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\models\social_lstm\train_4\training_parameters.json'
hyper_param_path = os.path.join(os.path.dirname(model_path), 'training_parameters.json')
patience = 8 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 32
lane_unit = 200  # each lane unit is 0.5 meters
time_span = 10
hidden_size = 64
social_size = 32
num_layers = 2
learning_rate = 1e-4
weight_decay = 1e-5
dropout = 0.3
num_epochs = 100

# Model initialization
model = ImprovedSocialLSTM(
    hidden_size=hidden_size,
    social_size=social_size,
    num_layers=num_layers,
    input_frames=time_span,
    output_size=2,  # Position and confidence
    dropout=dropout
).to(device)
model.load_state_dict(torch.load(model_path)['model_state_dict'])

C:\Users\zhChe\AppData\Local\Temp\ipykernel_26784\186032243.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path)['model_state_di

<All keys matched successfully>

In [56]:
train_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\train\train_dataset.h5')
val_dataset = TrajectoryDataset(r'D:\TimeSpaceDiagramDataset\SocialLSTMDataset\val\val_dataset.h5')
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=1)
for batch in val_dataset:
    inputs = batch['inputs'].to(device)
    targets = batch['targets'].to(device)
    break

In [58]:
for batch in train_loader:
    inputs = batch['inputs'].to(device)
    targets = batch['targets'].to(device)
    break

In [48]:
inputs.shape

torch.Size([4, 10, 3])

In [52]:
predictions

tensor([[-0.1377,  0.5353],
        [-0.1395,  0.5146],
        [-0.1563,  0.4963],
        [-0.1137,  0.5264]], device='cuda:0', grad_fn=<CatBackward0>)

In [53]:
targets

tensor([[140.],
        [ 30.],
        [137.],
        [ 26.]], device='cuda:0')

In [49]:
model.eval()
predictions = model(inputs)
